# Scout Quickstart Notebook

This notebook covers basics for querying Scout's HL7 radiology data via Trino + pandas. The `query()` helper below wraps SQLAlchemy and returns results as a DataFrame. For more advanced queries and tips on handling large results, see `Advanced.ipynb` in this same folder.

In [ ]:
# Scout's bundled query helper. Authenticates as you (per-user AuthZ
# applied through your Keycloak attributes) using the credentials your
# notebook session already holds. Use `:name` placeholders with values
# in `params`; for list values use Trino's `contains(array, element)` -
# SQLAlchemy doesn't expand list params into SQL IN clauses by default.
import scout

In [ ]:
# Define the facilities you have IRB approval for. 
# Queries below pass the facilities list as a parameter to filter the data to just those facilities.
FACILITIES = ['BJH', 'WUSM', 'BJCMG', 'SLCH']

## Exploring the data

Each row in `reports_latest` is the latest version of a radiology report. A patient may have many reports across time (priors, follow-ups, repeat imaging). To count distinct *patients* (not reports), use `reports_latest_epic_view`, which adds Scout's patient identifier `scout_patient_id`. See Scout's [data schema documentation](https://washu-scout.readthedocs.io/en/latest/dataschema.html) for more details on the available tables and columns.

In [ ]:
# Count the number of reports in Scout for your IRB-approved facilities.
total_reports = scout.query("""
    SELECT COUNT(*) AS reports
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
""", params={"facilities": FACILITIES})
total_reports

In [ ]:
# Count the number of distinct patients in Scout for your IRB-approved facilities
# using the `reports_latest_epic_view`, which adds Scout's patient identifier `scout_patient_id`.
total_patients = scout.query("""
    SELECT COUNT(DISTINCT scout_patient_id) AS patients
    FROM reports_latest_epic_view
    WHERE contains(:facilities, sending_facility)
""", params={"facilities": FACILITIES})
total_patients

In [ ]:
# Report counts broken down by facility.
reports_per_facility = scout.query("""
    SELECT sending_facility, COUNT(*) AS reports
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
    GROUP BY sending_facility
    ORDER BY reports DESC
""", params={"facilities": FACILITIES})
reports_per_facility

In [ ]:
# Distinct patient counts broken down by facility using `reports_latest_epic_view` and `scout_patient_id`.
patients_per_facility = scout.query("""
    SELECT sending_facility, COUNT(DISTINCT scout_patient_id) AS patients
    FROM reports_latest_epic_view
    WHERE contains(:facilities, sending_facility)
    GROUP BY sending_facility
    ORDER BY patients DESC
""", params={"facilities": FACILITIES})
patients_per_facility

In [ ]:
# Modality breakdown.
modality_count = scout.query("""
    SELECT modality, COUNT(*) AS reports
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
    GROUP BY modality
    ORDER BY reports DESC
""", params={"facilities": FACILITIES})
modality_count

In [ ]:
# Reports per year.
reports_per_year = scout.query("""
    SELECT year, COUNT(*) AS reports
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
    GROUP BY year
    ORDER BY year
""", params={"facilities": FACILITIES})
reports_per_year

## Querying reports

Filter reports by structured fields (modality, year, service_name), search free-text content (`report_text` or parsed sections like `report_section_impression`), or filter by diagnosis codes.

In [ ]:
# Pull one sample report to see available columns and get a sense of the data.
sample_report = scout.query("""
    SELECT *
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
    LIMIT 1
""", params={"facilities": FACILITIES})
sample_report

In [ ]:
# Filter reports by year. The `year` column is the partition column, so filtering on it should improve performance.
reports_recent = scout.query("""
    SELECT accession_number, modality, service_name, requested_dt, sending_facility
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
      AND year BETWEEN 2020 AND 2025
    LIMIT 100
""", params={"facilities": FACILITIES})
reports_recent

In [ ]:
# Filter by modality + body part (regex on service_name) + age + year.
combined_search = scout.query("""
    SELECT accession_number, patient_age, sex, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
      AND modality = 'MR'
      AND regexp_like(service_name, '(?i)(head|brain)')
      AND patient_age >= 18
      AND year = 2024
    LIMIT 100
""", params={"facilities": FACILITIES})
combined_search

In [ ]:
# Find reports for pulmonary embolism by searching for both a free-text mention in report_text OR an I26 diagnosis code on the report.
pe_search = scout.query("""
    SELECT accession_number, modality, service_name, requested_dt,
           report_section_impression, diagnoses
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
      AND year = 2024
      AND (
          regexp_like(report_text, '(?i)pulmonary embolism')
          OR any_match(diagnoses, d -> d.diagnosis_code LIKE 'I26%')
      )
    LIMIT 100
""", params={"facilities": FACILITIES})
pe_search

## Looking up a specific patient

Pull all reports for one patient by their Epic MRN. The `resolved_epic_mrn` column on `reports_latest_epic_view` bridges IDs across HL7 versions, so a single MRN catches reports where the MRN was inferred rather than carried on the message. See [Longitudinal Patient ID Resolution](https://washu-scout.readthedocs.io/en/latest/dataschema.html#longitudinal-patient-id-resolution) for more details.

In [ ]:
patient_mrn = "REPLACE_WITH_REAL_MRN"  # paste a real Epic MRN here

patient_history = scout.query("""
    SELECT resolved_epic_mrn AS epic_mrn, resolved_mpi AS mpi,
           requested_dt, modality, service_name, accession_number,
           report_section_impression
    FROM reports_latest_epic_view
    WHERE resolved_epic_mrn = :mrn
      AND contains(:facilities, sending_facility)
    ORDER BY requested_dt
""", params={"mrn": patient_mrn, "facilities": FACILITIES})
patient_history

## Bulk patient lookup from a CSV

Upload a CSV of Epic MRNs to `/home/jovyan/patient_ids.csv` and pull all their reports via `reports_latest_epic_view`. The `resolved_epic_mrn` column on the view bridges patient IDs across HL7 versions, so a single MRN catches reports where the MRN was inferred rather than carried on the message.

In [ ]:
patient_ids_path = '/home/jovyan/patient_ids.csv'
if not os.path.exists(patient_ids_path):
    pd.DataFrame({'epic_mrn': ['REPLACE_WITH_REAL_MRN']}).to_csv(patient_ids_path, index=False)
    print(f"Created sample CSV at {patient_ids_path} — edit with real MRNs")

patient_ids_df = pd.read_csv(patient_ids_path, dtype=str)
mrns = patient_ids_df['epic_mrn'].dropna().str.strip().tolist()
print(f"Loaded {len(mrns)} patient IDs from CSV")

In [ ]:
search = scout.query("""
    SELECT accession_number, version_id, epic_mrn, resolved_epic_mrn,
           modality, service_name, requested_dt, report_text
    FROM reports_latest_epic_view
    WHERE contains(:mrns, resolved_epic_mrn)
      AND contains(:facilities, sending_facility)
    ORDER BY resolved_epic_mrn, requested_dt
""", params={"mrns": mrns, "facilities": FACILITIES})
search.head()

In [ ]:
# Write results to CSV.
search.to_csv('/home/jovyan/patient_list_results.csv', index=False)

## Installing Additional Packages

The base Jupyter environment includes Trino client, pandas, matplotlib, seaborn, scikit-learn, statsmodels, pyarrow, and other core data analysis packages. For ML, NLP, or other specialized work, create a new conda environment:

```bash
# Create an environment with specific packages
mamba create -n my-env python=3.11 ipykernel pytorch transformers scikit-learn -y

# Or use the sample environment file (in ~/Scout/environment.yml)
mamba env create -f ~/Scout/environment.yml
```

**Important:** Include `ipykernel` in every environment you create. The `nb_conda_kernels` extension uses it to discover the environment as a Jupyter kernel. Without it, your environment won't appear in the kernel list.

After creating an environment, refresh the launcher to see it as a selectable kernel.

Environments are stored on your persistent home directory (`/home/jovyan/.conda/envs/`) and survive server restarts. In air-gapped deployments, package requests are routed through a proxy transparently — no extra configuration needed.

For more details, see the [Tips & Best Practices](https://washu-scout.readthedocs.io/en/latest/tips.html#installing-additional-packages) documentation.